# Modelos, métricas e análise de erros

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_08/03_modelos_metricas_e_erros.ipynb)

Regressão logística e árvore podem classificar, mas aprendem regras distintas. A
primeira combina evidências numa fronteira linear; a segunda cria decisões
sucessivas. Aqui usamos apenas metadados para tornar essas escolhas visíveis.

![A matriz separa verdadeiros e falsos positivos e negativos.](imagens/03_matriz_confusao.svg)

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_08'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('sklearn', 'scikit-learn>=1.5,<2')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

dados = pd.read_csv("dados/documentos_classificacao.csv")
treino, teste = train_test_split(dados, test_size=.25, random_state=42, stratify=dados["classe_alvo"])
colunas = ["periodo", "origem"]
preparo = ColumnTransformer([("categorias", OneHotEncoder(handle_unknown="ignore"), colunas)])
modelos = {
    "logística": Pipeline([("preparo", preparo), ("modelo", LogisticRegression(max_iter=1000, random_state=42))]),
    "árvore": Pipeline([("preparo", preparo), ("modelo", DecisionTreeClassifier(max_depth=3, min_samples_leaf=8, random_state=42))]),
}

def medir(y, previsto):
    return {"acurácia": accuracy_score(y, previsto), "precisão": precision_score(y, previsto, pos_label="relevante", zero_division=0), "revocação": recall_score(y, previsto, pos_label="relevante", zero_division=0), "F1": f1_score(y, previsto, pos_label="relevante", zero_division=0)}

previsoes = {}; resultados = []
for nome, modelo in modelos.items():
    modelo.fit(treino[colunas], treino["classe_alvo"])
    previsoes[nome] = modelo.predict(teste[colunas])
    resultados.append({"modelo": nome, **medir(teste["classe_alvo"], previsoes[nome])})
tabela_metricas = pd.DataFrame(resultados).set_index("modelo")
display(tabela_metricas.round(3))

## 2. Métricas respondem a perguntas diferentes

- **Acurácia:** parcela total correta.
- **Precisão:** entre os previstos como relevantes, quantos eram relevantes?
- **Revocação:** entre os relevantes, quantos foram encontrados?
- **F1:** média harmônica de precisão e revocação.

Escolher uma métrica é decidir qual erro importa. Em triagem documental, falso
negativo pode ocultar uma fonte; em publicação automática, falso positivo pode ser
mais danoso. A matriz de confusão conserva os quatro tipos de resultado.

In [ ]:
melhor_nome = tabela_metricas["F1"].idxmax()
melhor_previsto = previsoes[melhor_nome]
ConfusionMatrixDisplay.from_predictions(teste["classe_alvo"], melhor_previsto, labels=["não relevante", "relevante"], cmap="Blues")
plt.title(f"Matriz de confusão — {melhor_nome} — teste reservado")
plt.tight_layout(); plt.show()
erros = teste.assign(previsto=melhor_previsto)
erros = erros[erros["classe_alvo"] != erros["previsto"]]
display(erros[["id_documento", "periodo", "origem", "texto", "classe_alvo", "previsto"]].head(10))

## 3. Desempenho entre grupos

Uma média pode ocultar dano concentrado. Calculamos métricas por período no mesmo
teste. Grupos pequenos produzem estimativas instáveis; o resultado é uma auditoria
descritiva, não certificado de equidade.

In [ ]:
auditoria = []
avaliacao = teste.assign(previsto=melhor_previsto)
for periodo, grupo in avaliacao.groupby("periodo"):
    auditoria.append({"periodo": periodo, "n": len(grupo), **medir(grupo["classe_alvo"], grupo["previsto"])})
metricas_grupo = pd.DataFrame(auditoria).set_index("periodo")
display(metricas_grupo.round(3))

### U08-A04 — Atividade integrada — escolher pelo custo do erro

Escolha logística ou árvore após comparar linha de base, quatro métricas, matriz,
cinco erros e grupos. Declare qual erro é mais custoso para uma pergunta de
Humanidades e se a diferença entre grupos muda sua decisão.

**Dinâmica (35 minutos):** grupos de três assumem papéis de pesquisador, responsável
pelo acervo e pessoa afetada pela decisão. Produzam uma escolha e uma objeção.

**Resposta:** escreva aqui.

## Passagem

Metadados sozinhos não leem o documento. O próximo notebook representa palavras
por TF-IDF, compara com a linha de base e preserva o retorno aos textos errados.